In [ ]:
name = "SEYED AMIR HAJ SEYED TAGHIA"
student_id = "99101405"

# 1.Adversarial attacks

In this homework, we will discuss adversarial attacks on deep image classification models. Deep Neural Networks are a very powerful tool to recognize patterns in data, and, for example, perform image classification on a human-level. However, we have not tested yet how robust these models actually are. Can we "trick" the model and find failure modes? Can we design images that the networks naturally classify incorrectly? Due to the high classification accuracy on unseen test data, we would expect that this can be difficult. However, in 2014, a research group at Google and NYU showed that deep CNNs can be easily fooled, just by adding some salient but carefully constructed noise to the images. For instance, take a look at the example below (figure credit - [Goodfellow et al.](https://arxiv.org/pdf/1412.6572.pdf)):

<center width="100%" style="padding: 20px"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial10/adversarial_example.svg?raw=1" width="550px"></center>

The image on the left is the original image from ImageNet, and a deep CNN classifies the image correctly as "panda" with a class likelihood of 57%. Nevertheless, if we add a little noise to every pixel of the image, the prediction of the model changes completely. Instead of a panda, our CNN tells us that the image contains a "gibbon" with the confidence of over 99%. For a human, however, these two images look exactly alike, and you cannot distinguish which one has noise added and which doesn't. While this first seems like a fun game to fool trained networks, it can have a serious impact on the usage of neural networks. More and more deep learning models are used in applications, such as for example autonomous driving. Imagine that someone who gains access to the camera input of the car, could make pedestrians "disappear" for the image understanding network by simply adding some noise to the input as shown below (the figure is taken from [J.H. Metzen et al.](https://openaccess.thecvf.com/content_ICCV_2017/papers/Metzen_Universal_Adversarial_Perturbations_ICCV_2017_paper.pdf)). The first row shows the original image with the semantic segmentation output on the right (pedestrians red), while the second row shows the image with small noise and the corresponding segmentation prediction. The pedestrian becomes invisible for the network, and the car would think the road is clear ahead.

<center width="100%" style="padding: 20px"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial10/adversarial_attacks_cityscapes_3.png?raw=1" width="600px"></center>

Some attack types don't even require to add noise, but minor changes on a stop sign can be already sufficient for the network to recognize it as a "50km/h" speed sign ([paper](https://arxiv.org/pdf/1707.08945.pdf), [paper](https://arxiv.org/pdf/1802.06430.pdf)). The consequences of such attacks can be devastating. Hence, every deep learning engineer who designs models for an application should be aware of the possibility of adversarial attacks.

To understand what makes CNNs vulnerable to such attacks, we will implement our own adversarial attack strategies in this notebook, and try to fool a deep neural network. Let's being with importing our standard libraries:

In [ ]:
## Standard libraries
import os
import json
import math
import time
import numpy as np
import scipy.linalg

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgb
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0
import seaborn as sns
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
# Torchvision
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
# PyTorch Lightning
try:
    import pytorch_lightning as pl
except ModuleNotFoundError: # Google Colab does not have PyTorch Lightning installed by default. Hence, we do it here if necessary
    !pip install --quiet pytorch-lightning>=1.4
    import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

# Path to the folder where the datasets are/should be downloaded
DATASET_PATH = "/content"  # Update if necessary
CHECKPOINT_PATH = "/content/drive/MyDrive/models"

# Setting the seed
pl.seed_everything(42)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

We have again a few download statements. This includes both a dataset, and a few pretrained patches we will use later.

# 10 Points

In [ ]:
# === STEP 0: Install & Authenticate Kaggle ===
!pip install -q kaggle

import os, json, shutil, glob, random, torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# === STEP 1: Kaggle API credentials ===
kaggle_token = {
    "username": "frdjjnlnnln",
    "key": "526b3017044d9e07ae4c84c4fb89c714"
}
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_token, f)
os.chmod("/root/.kaggle/kaggle.json", 600)

# === STEP 2: Download ImageNet-10K ===
!kaggle datasets download -d priyerana/imagenet-10k -p ./imagenet10k
!unzip -q ./imagenet10k/imagenet-10k.zip -d ./imagenet10k

# === STEP 3: Extract Synset IDs from LOC_synset_mapping.txt ===
target_names = [
    'toaster', 'goldfish', 'school bus', 'lipstick', 'pineapple',
    'banana', 'snail', 'acoustic guitar', 'airliner', 'zebra'
]
mapping_path = '/content/LOC_synset_mapping.txt'  # ← آپلود کن توی Colab

# Load mapping
name_to_synset = {}
with open(mapping_path, 'r') as f:
    for line in f:
        synset_id, name = line.strip().split(' ', 1)
        for n in name.split(', '):
            name_to_synset[n.strip()] = synset_id

target_synsets = [name_to_synset[name] for name in target_names]
print("✅ Synset IDs:", target_synsets)

# === STEP 4: Copy only selected classes ===
source_dir = './imagenet10k/imagenet_subtrain'
target_dir = './imagenet10k_subset_10'
os.makedirs(target_dir, exist_ok=True)

for syn in target_synsets:
    src = os.path.join(source_dir, syn)
    dst = os.path.join(target_dir, syn)
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f"✅ Copied class: {syn}")
        else:
            print(f"🔁 Already exists, skipping: {syn}")
    else:
        print(f"⚠️ Not found in source: {syn}")

# === STEP 5: Load Dataset ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(target_dir, transform=transform)
print(f"📦 Total samples: {len(full_dataset)} | Classes: {len(full_dataset.classes)}")

# === STEP 6: Split into Train/Test ===
subset_size = min(2000, len(full_dataset))
random.seed(42)
all_indices = random.sample(range(len(full_dataset)), subset_size)
train_indices, test_indices = train_test_split(all_indices, test_size=0.2, random_state=42)

train_dataset = Subset(full_dataset, train_indices)
test_dataset = Subset(full_dataset, test_indices)

## Deep CNNs on ImageNet

For our experiments in this notebook, we will use common CNN architectures trained on the ImageNet dataset. Such models are luckily provided by PyTorch's torchvision package, and hence we just need to load the model of our preference. For the results  default on Google Colab, we use a ResNet34.

# 5 Points

In [ ]:
import torch
import torchvision.models as models

# Load the pre-trained ResNet34 model
resnet34 = models.resnet34(pretrained=True)

# Set the model to evaluation mode (important for inference)
resnet34.eval()

# No gradients are needed for the network (important for inference)
for param in resnet34.parameters():
    param.requires_grad = False

# Check if CUDA is available and move the model to the GPU if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet34.to(device)

# Print the model architecture to confirm it's loaded correctly
print(resnet34)

To perform adversarial attacks, we also need a dataset to work on. Given that the CNN model has been trained on ImageNet, it is only fair to perform the attacks on data from ImageNet. For this, we provide a small set of pre-processed images from the original ImageNet dataset . Specifically, we have 10 images for each of the 1000 labels of the dataset. We can load the data below, and create a corresponding data loader.

# 5 Points

In [ ]:
full_dataset = datasets.ImageFolder(target_dir, transform=transform)
print(f"📦 Total samples: {len(full_dataset)} | Classes: {len(full_dataset.classes)}")

# === STEP 6: Split into Train/Test ===
subset_size = min(2000, len(full_dataset))
random.seed(42)
all_indices = random.sample(range(len(full_dataset)), subset_size)
train_indices, test_indices = train_test_split(all_indices, test_size=0.2, random_state=42)

train_dataset = Subset(full_dataset, train_indices)
test_dataset = Subset(full_dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

Before we start with our attacks, we should verify the performance of our model. As ImageNet has 1000 classes, simply looking at the accuracy is not sufficient to tell the performance of a model. Imagine a model that always predicts the true label as the second-highest class in its softmax output. Although we would say it recognizes the object in the image, it achieves an accuracy of 0. In ImageNet with 1000 classes, there is not always one clear label we can assign an image to. This is why for image classifications over so many classes, a common alternative metric is "Top-5 accuracy", which tells us how many times the true label has been within the 5 most-likely predictions of the model. As models usually perform quite well on those, we report the error (1 - accuracy) instead of the accuracy:

# 10 Points

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In this section, I load the dataset and create a subset containing 10 classes, including the 5 classes required for the subsequent code. Then, I fine-tune the ResNet model, which achieved an accuracy error of 5% for both top-1 and top-5 metrics

In [ ]:
# === STEP 0: Install & Authenticate Kaggle ===
!pip install -q kaggle

import os, json, shutil, glob, random, torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# === STEP 1: Kaggle API credentials ===
kaggle_token = {
    "username": "frdjjnlnnln",
    "key": "526b3017044d9e07ae4c84c4fb89c714"
}
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_token, f)
os.chmod("/root/.kaggle/kaggle.json", 600)

# === STEP 2: Download ImageNet-10K ===
!kaggle datasets download -d priyerana/imagenet-10k -p ./imagenet10k
!unzip -q ./imagenet10k/imagenet-10k.zip -d ./imagenet10k

# === STEP 3: Extract Synset IDs from LOC_synset_mapping.txt ===
target_names = [
    'toaster', 'goldfish', 'school bus', 'lipstick', 'pineapple',
    'banana', 'snail', 'acoustic guitar', 'airliner', 'zebra'
]
mapping_path = '/content/LOC_synset_mapping.txt'  # ← آپلود کن توی Colab

# Load mapping
name_to_synset = {}
with open(mapping_path, 'r') as f:
    for line in f:
        synset_id, name = line.strip().split(' ', 1)
        for n in name.split(', '):
            name_to_synset[n.strip()] = synset_id

target_synsets = [name_to_synset[name] for name in target_names]
print("✅ Synset IDs:", target_synsets)

# === STEP 4: Copy only selected classes ===
source_dir = './imagenet10k/imagenet_subtrain'
target_dir = './imagenet10k_subset_10'
os.makedirs(target_dir, exist_ok=True)

for syn in target_synsets:
    src = os.path.join(source_dir, syn)
    dst = os.path.join(target_dir, syn)
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f"✅ Copied class: {syn}")
        else:
            print(f"🔁 Already exists, skipping: {syn}")
    else:
        print(f"⚠️ Not found in source: {syn}")

# === STEP 5: Load Dataset ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(target_dir, transform=transform)
print(f"📦 Total samples: {len(full_dataset)} | Classes: {len(full_dataset.classes)}")

# === STEP 6: Split into Train/Test ===
subset_size = min(2000, len(full_dataset))
random.seed(42)
all_indices = random.sample(range(len(full_dataset)), subset_size)
train_indices, test_indices = train_test_split(all_indices, test_size=0.2, random_state=42)

train_dataset = Subset(full_dataset, train_indices)
test_dataset = Subset(full_dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# === STEP 7: Load ResNet34 WITHOUT Fine-Tuning ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(full_dataset.classes)

resnet34 = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
for param in resnet34.parameters():
    param.requires_grad = False

resnet34.fc = nn.Linear(resnet34.fc.in_features, num_classes)
resnet34 = resnet34.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet34.fc.parameters(), lr=1e-3)

# === STEP 8: Training Loop ===
num_epochs = 15
for epoch in range(num_epochs):
    resnet34.train()
    total_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet34(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # === Evaluate on test set ===
    resnet34.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = resnet34(imgs)
            _, pred_top1 = torch.max(outputs, 1)
            _, pred_top5 = torch.topk(outputs, 5, dim=1)
            correct_top1 += (pred_top1 == labels).sum().item()
            correct_top5 += (pred_top5 == labels.view(-1, 1)).sum().item()
            total += labels.size(0)

    top1_acc = 100 * correct_top1 / total
    top5_acc = 100 * correct_top5 / total

    print(f"📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss     : {avg_loss:.4f}")
    print(f"   Test Top-1 Acc : {top1_acc:.2f}%")
    print(f"   Test Top-5 Acc : {top5_acc:.2f}%\\n")

# === STEP 9: Save Model ===
save_dir = "/content/drive/MyDrive/models"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, "resnet34_no_ft_10class.pth")
torch.save(resnet34.state_dict(), model_path)
print(f"✅ Model saved to: {model_path}")


In this section, I tested the ResNet model on the CIFAR-10 dataset, achieving a respectable accuracy error rate.

In [ ]:
# Step 0: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create directory in Google Drive (optional)
save_dir = '/content/drive/MyDrive/models'
os.makedirs(save_dir, exist_ok=True)

# Step 1: Imports and Setup
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
import random
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Step 2: CIFAR-10 Transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet expects 224x224
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Step 3: Download CIFAR-10 Dataset
train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
val_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Step 4: Subsample for faster training (optional)
random.seed(42)
train_subset = Subset(train_dataset, random.sample(range(len(train_dataset)), 10000))
val_subset = Subset(val_dataset, random.sample(range(len(val_dataset)), 2000))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_subset)} | Val samples: {len(val_subset)}")

# Step 5: Load Pretrained ResNet34 and Adjust for CIFAR-10
resnet34 = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
resnet34.fc = nn.Linear(resnet34.fc.in_features, 10)  # CIFAR-10 has 10 classes
resnet34 = resnet34.to(device)

# Step 6: Training Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet34.parameters(), lr=1e-3)

# Step 7: Training and Validation
num_epochs = 20
best_acc = 0.0
model_path = os.path.join(save_dir, "resnet34_cifar10.pth")

for epoch in range(num_epochs):
    resnet34.train()
    total_train_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} - Train"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet34(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    resnet34.eval()
    total_val_loss = 0
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = resnet34(imgs)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()
            _, pred_top1 = torch.max(outputs, 1)
            _, pred_top5 = torch.topk(outputs, 5, dim=1)
            correct_top1 += (pred_top1 == labels).sum().item()
            correct_top5 += (pred_top5 == labels.view(-1, 1)).sum().item()
            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader)
    top1_acc = 100 * correct_top1 / total
    top5_acc = 100 * correct_top5 / total

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Train Loss     : {avg_train_loss:.4f}")
    print(f"Val Loss       : {avg_val_loss:.4f}")
    print(f"Top-1 Accuracy : {top1_acc:.2f}%")
    print(f"Top-5 Accuracy : {top5_acc:.2f}%\n")

    # Save model if it achieves better top-1 accuracy
    if top1_acc > best_acc:
        best_acc = top1_acc
        torch.save(resnet34.state_dict(), model_path)
        print(f"✅ Best model saved to {model_path}")


The ResNet34 achives a decent error rate of 5% for the top-5 predictions. Next, we can look at some predictions of the model to get more familiar with the dataset. The function below plots an image along with a bar diagram of its predictions. We also prepare it to show adversarial examples for later applications.

# 15 Points

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

def show_prediction(img, label, pred, K=5, adv_img=None, noise=None, class_names=None):
    """
    Visualizes the original image, top-K predictions, optional adversarial image, and noise.

    Parameters:
    - img: Original image (Tensor: 3xHxW)
    - label: Ground truth label (int or str)
    - pred: Tuple of (probabilities, class indices) from topk()
    - K: Number of top predictions to show
    - adv_img: (Optional) Adversarial image (Tensor)
    - noise: (Optional) Perturbation/noise (Tensor)
    - class_names: (Optional) List of class names
    """
    # Undo normalization
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    def denormalize(t):
        t = t.clone().cpu().detach().numpy()
        t = np.transpose(t, (1, 2, 0))  # CxHxW to HxWxC
        t = std * t + mean
        return np.clip(t, 0, 1)

    img_np = denormalize(img)

    fig, axes = plt.subplots(1, 3 if noise is not None else 2, figsize=(14, 5))

    # Original Image
    ax = axes[0]
    ax.imshow(img_np)
    title = f"True: {label}" if isinstance(label, str) else f"Label: {label}"
    ax.set_title(title, fontsize=14)
    ax.axis('off')

    # Predictions
    ax = axes[1]
    probs, indices = pred
    top_probs = probs[:K].cpu().numpy()
    top_classes = [class_names[i] if class_names else f"Class {i}" for i in indices[:K].cpu().numpy()]
    ax.barh(top_classes[::-1], top_probs[::-1])
    ax.set_title("Top Predictions", fontsize=14)
    ax.set_xlim(0, 1)

    # Noise or Adversarial Image
    if noise is not None:
        ax = axes[2]
        noise_np = denormalize(noise)
        ax.imshow(noise_np)
        ax.set_title("Noise / Perturbation", fontsize=14)
        ax.axis('off')
    elif adv_img is not None:
        ax = axes[2]
        adv_np = denormalize(adv_img)
        ax.imshow(adv_np)
        ax.set_title("Adversarial Image", fontsize=14)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


Let's visualize a few images below:

first i loaded the saved model

In [ ]:
# Step 1: Load saved model from Google Drive
import torch
import torchvision.models as models

model_path = "/content/drive/MyDrive/models/resnet34_no_ft_10class.pth"

resnet34 = models.resnet34(weights=None)  # no pretrained weights
resnet34.fc = torch.nn.Linear(resnet34.fc.in_features, 10)
resnet34.load_state_dict(torch.load(model_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
resnet34 = resnet34.to(device)
resnet34.eval()
print("✅ Model loaded from:", model_path)


In [ ]:
class_names = [
    'toaster', 'goldfish', 'school bus', 'lipstick', 'pineapple',
    'banana', 'snail', 'acoustic guitar', 'airliner', 'zebra'
]


In [ ]:
import torch.nn.functional as F

# === Prepare class names (synset ID → readable name) ===
synset_to_name = {v: k for k, v in name_to_synset.items()}
class_names = [synset_to_name[syn] for syn in full_dataset.classes]

# === Get a batch from test set ===
exmp_batch, label_batch = next(iter(test_loader))
exmp_batch = exmp_batch.to(device)
label_batch = label_batch.to(device)

# === Forward pass ===
with torch.no_grad():
    outputs = resnet34(exmp_batch)
    probs = F.softmax(outputs, dim=1)
    top_probs, top_indices = torch.topk(probs, k=5, dim=1)

# === Visualize using your show_prediction function ===
for i in range(min(10, exmp_batch.size(0))):  # Show first 10
    show_prediction(
        img=exmp_batch[i],
        label=class_names[label_batch[i].item()],
        pred=(top_probs[i], top_indices[i]),
        K=5,
        adv_img=None,    # optional
        noise=None,      # optional
        class_names=class_names
    )


The bar plot on the right shows the top-5 predictions of the model with their class probabilities. We denote the class probabilities with "confidence" as it somewhat resembles how confident the network is that the image is of one specific class. Some of the images have a highly peaked probability distribution, and we would expect the model to be rather robust against noise for those. However, we will see below that this is not always the case.

## White-box adversarial attacks

There have been proposed many possible adversarial attack strategies, which all share the same goal: alternate the data/image input only a little bit to have a great impact on the model's prediction. Specifically, if we look at the ImageNet predictions above, how can we have to change the image of the goldfish so that the model does not recognize the goldfish anymore? At the same time, the label of the image should not change, in the sense that a human would still clearly classify it as a goldfish. This is the same objective that the generator network has in the Generative Adversarial Network framework: try to fool another network (discriminator) by changing its input.

Adversarial attacks are usually grouped into "white-box" and "black-box" attacks. White-box attacks assume that we have access to the model parameter and can, for example, calculate the gradients with respect to the input (similar as in GANs). Black-box attacks on the other hand have the harder task of not having any knowledge about the network, and can only obtain predictions for an image, but no gradients or the like. In this notebook, we will focus on white-box attacks as they are usually easier to implement and follow the intuition of Generative Adversarial Networks (GAN) as studied in lecture 10.

### Fast Gradient Sign Method (FGSM)

One of the first attack strategies proposed is Fast Gradient Sign Method (FGSM), developed by [Ian Goodfellow et al.](https://arxiv.org/pdf/1412.6572.pdf) in 2014. Given an image, we create an adversarial example by the following expression:

$$\tilde{x} = x + \epsilon \cdot \text{sign}(\nabla_x J(\theta,x,y))$$

The term $J(\theta,x,y)$ represents the loss of the network for classifying input image $x$ as label $y$; $\epsilon$ is the intensity of the noise, and $\tilde{x}$ the final adversarial example. The equation resembles SGD and is actually nothing else than that. We change the input image $x$ in the direction of *maximizing* the loss $J(\theta,x,y)$. This is exactly the other way round as during training, where we try to minimize the loss. The sign function and $\epsilon$ can be seen as gradient clipping and learning rate specifically. We only allow our attack to change each pixel value by $\epsilon$. You can also see that the attack can be performed very fast, as it only requires a single forward and backward pass. Let's implement it below:

# 10 Points

In [ ]:
def fast_gradient_sign_method(model, imgs, labels, epsilon=0.02, device='cuda'):
    """
    Generate adversarial examples using FGSM in normalized space.

    Args:
        model: Trained model (in eval mode)
        imgs: Normalized input images [B, 3, H, W]
        labels: True labels (long tensor) [B]
        epsilon: Perturbation magnitude
        device: 'cuda' or 'cpu'

    Returns:
        adv_imgs: Adversarial images (still normalized)
        noise: Perturbation added
    """
    model.eval()
    imgs = imgs.clone().detach().to(device)
    imgs.requires_grad = True
    labels = labels.to(device)

    # Forward + backward
    outputs = model(imgs)
    loss = F.cross_entropy(outputs, labels)
    model.zero_grad()
    loss.backward()

    # Create perturbation and adversarial image
    noise = imgs.grad.data.sign()
    adv_imgs = imgs + epsilon * noise

    return adv_imgs.detach(), (epsilon * noise).detach()


The default value of $\epsilon=0.02$ corresponds to changing a pixel value by about 1 in the range of 0 to 255, e.g. changing 127 to 128. This difference is marginal and can often not be recognized by humans. Let's try it below on our example images:

In [ ]:
# Get a test batch
exmp_batch, label_batch = next(iter(test_loader))
exmp_batch = exmp_batch.to(device)
label_batch = label_batch.to(device)

# Generate adversarial examples
adv_imgs, noise = fast_gradient_sign_method(
    model=resnet34,
    imgs=exmp_batch,
    labels=label_batch,
    epsilon=0.02,
    device=device
)

# Predict on adversarial examples
with torch.no_grad():
    adv_outputs = resnet34(adv_imgs)
    adv_probs = F.softmax(adv_outputs, dim=1)
    adv_top_probs, adv_top_indices = torch.topk(adv_probs, k=5, dim=1)

# Visualize predictions
for i in range(min(8, exmp_batch.size(0))):
    show_prediction(
        img=exmp_batch[i],
        label=class_names[label_batch[i].item()],
        pred=(adv_top_probs[i], adv_top_indices[i]),
        adv_img=adv_imgs[i],
        noise=noise[i],
        class_names=class_names
    )


Despite the minor amount of noise, we are able to fool the network on all of our examples. None of the labels have made it into the top-5 for the four images, showing that we indeed fooled the model. We can also check the accuracy of the model on the adversarial images:

In [ ]:
def eval_model(model, data_loader, img_func=None, device='cuda'):
    """
    Evaluate a model on a dataset (clean or adversarial).

    Args:
        model: PyTorch model (e.g., resnet34)
        data_loader: PyTorch DataLoader (e.g., test_loader)
        img_func: Optional function(imgs, labels) → adversarial imgs
        device: 'cuda' or 'cpu'

    Returns:
        top1_acc, top5_acc: accuracy percentages
    """
    model.eval()
    total = 0
    correct_top1 = 0
    correct_top5 = 0

    for imgs, labels in data_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if img_func is not None:
            imgs = img_func(imgs, labels)

        with torch.no_grad():
            outputs = model(imgs)
            _, pred_top1 = torch.max(outputs, dim=1)
            _, pred_top5 = torch.topk(outputs, 5, dim=1)

        correct_top1 += (pred_top1 == labels).sum().item()
        correct_top5 += (pred_top5 == labels.view(-1, 1)).sum().item()
        total += labels.size(0)

    top1_acc = 100 * correct_top1 / total
    top5_acc = 100 * correct_top5 / total

    print(f"\n📊 Evaluation Results:")
    print(f"✅ Top-1 Accuracy : {top1_acc:.2f}%")
    print(f"✅ Top-5 Accuracy : {top5_acc:.2f}%")
    print(f"❌ Top-1 Error    : {100 - top1_acc:.2f}%")
    print(f"❌ Top-5 Error    : {100 - top5_acc:.2f}%\n")

    return top1_acc, top5_acc


In [ ]:
# Clean accuracy
eval_model(resnet34, test_loader, device=device)

# Adversarial accuracy with FGSM
eval_model(
    model=resnet34,
    data_loader=test_loader,
    img_func=lambda x, y: fast_gradient_sign_method(resnet34, x, y, epsilon=0.02, device=device)[0],
    device=device
)


"As expected, the model is misled on nearly every image, especially in terms of the Top-1 error, with an error rate of 75.00%. Furthermore, more than half of the predictions do not have the true label within the top-5, with a Top-5 error rate of 35.00%. This is a significant contrast to the 5% error rate observed on clean images. However, the predictions are still semantically similar. For example, in the images we visualized above, the tench is still recognized as another type of fish, and the great white shark is misclassified as a dugong. FGSM could be adapted to increase the probability of a specific class rather than minimizing the probability of a label, but for these types of attacks, alternatives like the adversarial patch tend to be more effective.

### Adversarial Patches

Instead of changing every pixel by a little bit, we can also try to change a small part of the image into whatever values we would like. In other words, we will create a small image patch that covers a minor part of the original image but causes the model to confidentially predict a specific class we choose. This form of attack is an even bigger threat in real-world applications than FSGM. Imagine a network in an autonomous car that receives a live image from a camera. Another driver could print out a specific pattern and put it on the back of his/her vehicle to make the autonomous car believe that the car is actually a pedestrian. Meanwhile, humans would not notice it. [Tom Brown et al.](https://arxiv.org/pdf/1712.09665.pdf) proposed a way of learning such adversarial image patches robustly in 2017 and provided a short demonstration on [YouTube](https://youtu.be/i1sp4X57TL4). Interestingly, if you add a small picture of the target class (here *toaster*) to the original image, the model does not pick it up at all. A specifically designed patch, however, which only roughly looks like a toaster, can change the network's prediction instantaneously.

[![Adversarial patch in real world](https://img.youtube.com/vi/i1sp4X57TL4/0.jpg)](https://youtu.be/i1sp4X57TL4)

Let's take a closer look at how we can actually train such patches. The general idea is very similar to FSGM in the sense that we calculate gradients for the input, and update our adversarial input correspondingly. However, there are also some differences in the setup. Firstly, we do not calculate a gradient for every pixel. Instead, we replace parts of the input image with our patch and then calculate the gradients just for our patch. Secondly, we don't just do it for one image, but we want the patch to work with any possible image. Hence, we have a whole training loop where we train the patch using SGD. Lastly, image patches are usually designed to make the model predict a specific class, not just any other arbitrary class except the true label. For instance, we can try to create a patch for the class "toaster" and train the patch so that our pretrained model predicts the class "toaster" for any image with the patch in it.

Additionally, to the setup described above, there are a couple of design choices we can take. For instance, [Brown et al.](https://arxiv.org/pdf/1712.09665.pdf) randomly rotated and scaled the patch during training before placing it at a random position in an input image. This makes the patch more robust to small changes and is necessary if we want to fool a neural network in a real-world application. For simplicity, we will only focus on making the patch robust to the location in the image. Given a batch of input images and a patch, we can add the patch as follows:

# 5 Points

In [ ]:
import torch
import random

def place_patch(img, patch, top=None, left=None):
    img = img.clone()
    _, H, W = img.shape
    _, h, w = patch.shape

    # اگر مختصات مشخص نشده، به‌صورت وسط تصویر قرار بگیرد
    if top is None:
        top = (H - h) // 2
    if left is None:
        left = (W - w) // 2

    img[:, top:top + h, left:left + w] = patch
    return img



The patch itself will be an `nn.Parameter` whose values are in the range between $-\infty$ and $\infty$. Images are, however, naturally limited in their range, and thus we write a small function that maps the parameter into the image value range of ImageNet:

# 5 Points

In [ ]:
# ImageNet normalization constants
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TENSOR_MEANS = torch.FloatTensor(NORM_MEAN)[:, None, None]
TENSOR_STD = torch.FloatTensor(NORM_STD)[:, None, None]

def patch_forward(patch):
    patch_img = torch.sigmoid(patch)
    patch_normalized = (patch_img - TENSOR_MEANS.to(patch.device)) / TENSOR_STD.to(patch.device)
    return patch_normalized


Before looking at the actual training code, we can write a small evaluation function. We evaluate the success of a patch by how many times we were able to fool the network into predicting our target class. A simple function for this is implemented below.

# 10 Points

In [ ]:
def eval_patch(model, patch, data_loader, target_class_name, name_to_synset, full_dataset, device='cuda'):
    model.eval()
    target_syn = name_to_synset[target_class_name]
    target_class = full_dataset.class_to_idx[target_syn]
    fooled_top1, fooled_top5, total = 0, 0, 0

    for imgs, labels in data_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        mask = labels != target_class
        if mask.sum() == 0:
            continue
        imgs, labels = imgs[mask], labels[mask]

        pred_top1, pred_top5 = torch.zeros(len(imgs), dtype=torch.bool).to(device), torch.zeros(len(imgs), dtype=torch.bool).to(device)
        for _ in range(4):
            patched = torch.stack([place_patch(img, patch) for img in imgs])
            outputs = model(patched)
            top1 = outputs.argmax(1)
            top5 = torch.topk(outputs, 5, dim=1).indices
            pred_top1 |= (top1 == target_class)
            pred_top5 |= (top5 == target_class).any(dim=1)

        fooled_top1 += pred_top1.sum().item()
        fooled_top5 += pred_top5.sum().item()
        total += len(imgs)

    acc = 100 * fooled_top1 / total
    top5 = 100 * fooled_top5 / total
    print(f"\n🎯 Patch Attack Success (Excl. True '{target_class_name}'):\n   Top-1 → {acc:.2f}%\n   Top-5 → {top5:.2f}%")
    return acc, top5

Finally, we can look at the training loop. Given a model to fool, a target class to design the patch for, and a size $k$ of the patch in the number of pixels, we first start by creating a parameter of size $3\times k\times k$. These are the only parameters we will train, and the network itself remains untouched. We use a simple SGD optimizer with momentum to minimize the classification loss of the model given the patch in the image. While we first start with a very high loss due to the good initial performance of the network, the loss quickly decreases once we start changing the patch. In the end, the patch will represent patterns that are characteristic of the class. For instance, if we would want the model to predict a "goldfish" in every image, we would expect the pattern to look somewhat like a goldfish. Over the iterations, the model finetunes the pattern and, hopefully, achieves a high fooling accuracy.

# 15 Points

In [ ]:
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random
import torch
from torch.utils.data import DataLoader

def patch_attack_augmented(model, target_class_name, name_to_synset, full_dataset,
                            patch_size=64, num_epochs=10, device='cuda'):
    model.to(device)
    model.eval()

    target_syn = name_to_synset[target_class_name]
    target_class = full_dataset.class_to_idx[target_syn]

    example_img = next(img.to(device) for img, label in full_dataset if label == target_class)

    C, H, W = example_img.shape
    top = (H - patch_size) // 2
    left = (W - patch_size) // 2
    init_patch = example_img[:, top:top+patch_size, left:left+patch_size]
    init_patch_param = torch.logit(init_patch.clamp(1e-6, 1 - 1e-6))

    patch_param = torch.nn.Parameter(init_patch_param.clone().detach().to(device).requires_grad_())
    optimizer = torch.optim.Adam([patch_param], lr=1e-3, weight_decay=1e-4)

    val_size = 100
    train_data, val_data = torch.utils.data.random_split(train_dataset, [len(train_dataset)-val_size, val_size])
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

    def show_patch(title, patch_tensor):
        patch_vis = patch_tensor.detach().cpu().clone()
        patch_vis = torch.sigmoid(patch_vis) if patch_vis.max() > 1 else patch_vis
        patch_vis = torch.clamp(patch_vis, 0, 1).permute(1, 2, 0).numpy()
        plt.imshow(patch_vis)
        plt.title(title)
        plt.axis('off')
        plt.show()

    show_patch("🎨 Initial Patch", patch_forward(patch_param))

    for epoch in range(num_epochs):
        total_loss = 0
        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            patch = patch_forward(patch_param)
            patched_imgs = []

            for img in imgs:
                angle = random.uniform(-15, 15)
                scale = random.uniform(0.9, 1.1)
                p = TF.resize(patch, [int(patch_size * scale)] * 2)
                p = TF.rotate(p, angle)
                _, H, W = img.shape
                ph, pw = p.shape[1:]
                top = (H - ph) // 2
                left = (W - pw) // 2
                img_with_patch = img.clone()
                img_with_patch[:, top:top + ph, left:left + pw] = p
                patched_imgs.append(img_with_patch)

            patched_imgs = torch.stack(patched_imgs)
            target_labels = torch.full((patched_imgs.size(0),), target_class, dtype=torch.long, device=device)
            outputs = model(patched_imgs)
            loss = F.cross_entropy(outputs, target_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"📚 Epoch {epoch+1}/{num_epochs} - Avg Loss: {total_loss / len(train_loader):.4f}")

    patch_final = patch_forward(patch_param).detach()
    acc, top5 = eval_patch(
        model=model,
        patch=patch_final,
        data_loader=val_loader,
        target_class_name=target_class_name,
        name_to_synset=name_to_synset,
        full_dataset=full_dataset,
        device=device
    )

    img, _ = val_data[0]
    img = img.to(device)
    _, H, W = img.shape
    ph, pw = patch_final.shape[1:]
    top = (H - ph) // 2
    left = (W - pw) // 2
    img_with_patch = img.clone()
    img_with_patch[:, top:top + ph, left:left + pw] = patch_final

    def denormalize(t):
        mean = torch.tensor([0.485, 0.456, 0.406], device=t.device).view(3,1,1)
        std = torch.tensor([0.229, 0.224, 0.225], device=t.device).view(3,1,1)
        return torch.clamp((t * std + mean), 0, 1).permute(1, 2, 0).cpu().numpy()

    plt.imshow(denormalize(img_with_patch))
    plt.title("🔍 Patched Image Example")
    plt.axis('off')
    plt.show()

    show_patch("🎯 Final Learned Patch", patch_param)

    return patch_final, {"acc": acc, "top5": top5}


To get some experience with what to expect from an adversarial patch attack, we want to train multiple patches for different classes. As the training of a patch can take one or two minutes on a GPU, we have provided a couple of pre-trained patches including their results on the full dataset. The results are saved in a JSON file, which is loaded below.

In [ ]:
# Load evaluation results of the pretrained patches
json_results_file = os.path.join(CHECKPOINT_PATH, "patch_results.json")
json_results = {}
if os.path.isfile(json_results_file):
    with open(json_results_file, "r") as f:
        json_results = json.load(f)

# If you train new patches, you can save the results via calling this function
def save_results(patch_dict):
    result_dict = {cname: {psize: [t.item() if isinstance(t, torch.Tensor) else t
                                   for t in patch_dict[cname][psize]["results"]]
                           for psize in patch_dict[cname]}
                   for cname in patch_dict}
    with open(os.path.join(CHECKPOINT_PATH, "patch_results.json"), "w") as f:
        json.dump(result_dict, f, indent=4)

Additionally, we implement a function to train and evaluate patches for a list of classes and patch sizes. The pretrained patches include the classes *toaster*, *goldfish*, *school bus*, *lipstick*, and *pineapple*. We chose the classes arbitrarily to cover multiple domains (animals, vehicles, fruits, devices, etc.). We trained each class for three different patch sizes: $32\times32$ pixels, $48\times48$ pixels, and $64\times64$ pixels. We can load them in the two cells below.

In [ ]:
import os
import torch
import json

def get_patches(class_names, patch_sizes, model, name_to_synset, full_dataset,
                checkpoint_path, train_dataset, device='cuda', num_epochs=5):
    """
    Always trains new adversarial patches and re-evaluates them (ignores saved results).
    """

    os.makedirs(checkpoint_path, exist_ok=True)
    result_dict = {}

    for cname in class_names:
        print(f"\n🎯 Class: {cname}")
        result_dict[cname] = {}

        for psize in patch_sizes:
            print(f"  📦 Patch size: {psize}x{psize}")

            # Always train a new patch (overwrite if exists)
            print(f"    🔁 Re-training patch for {cname} ({psize}px)")
            raw_patch, metrics = patch_attack_augmented(
                model=model,
                target_class_name=cname,
                name_to_synset=name_to_synset,
                full_dataset=full_dataset,
                patch_size=psize,
                num_epochs=num_epochs,
                device=device
            )

            # Save new patch
            patch_path = os.path.join(checkpoint_path, f"{cname}_{psize}.pth")
            torch.save(raw_patch, patch_path)
            print(f"    💾 Saved patch to: {patch_path}")

            # Always evaluate the freshly trained patch
            result_dict[cname][psize] = {
                "patch": raw_patch,
                "results": [metrics["acc"], metrics["top5"]]
            }

    return result_dict


Feel free to add any additional classes and/or patch sizes.

In [ ]:
class_names = ['toaster', 'goldfish', 'school bus', 'lipstick', 'pineapple']
patch_sizes = [32,48,64,128]

patch_dict = get_patches(
    class_names=class_names,
    patch_sizes=patch_sizes,
    model=resnet34,
    name_to_synset=name_to_synset,
    full_dataset=full_dataset,
    checkpoint_path=CHECKPOINT_PATH,
    train_dataset=train_dataset,
    device=device,
    num_epochs=10
)

# ذخیره نتایج اگر کلاس جدید اضافه کردی
save_results(patch_dict)


Before looking at the quantitative results, we can actually visualize the patches.

# 5 Points

In [ ]:
import matplotlib.pyplot as plt

def show_patches(patch_dict):
    """
    نمایش تمام adversarial patchها برای کلاس‌ها و اندازه‌های مختلف.

    Args:
        patch_dict: خروجی get_patches — دیکشنری شامل patchها و نتایج آن‌ها
    """
    num_classes = len(patch_dict)
    patch_sizes = sorted(list(next(iter(patch_dict.values())).keys()))
    num_sizes = len(patch_sizes)

    fig, axes = plt.subplots(num_classes, num_sizes, figsize=(4 * num_sizes, 3.5 * num_classes))

    if num_classes == 1:
        axes = [axes]
    if num_sizes == 1:
        axes = [[ax] for ax in axes]

    for i, (class_name, sizes) in enumerate(patch_dict.items()):
        for j, psize in enumerate(patch_sizes):
            patch = patch_dict[class_name][psize]["patch"].cpu().detach()
            patch = torch.sigmoid(patch)  # map from (-inf, inf) to [0, 1]
            patch_img = patch.permute(1, 2, 0).numpy()

            ax = axes[i][j]
            ax.imshow(patch_img)
            acc, top5 = patch_dict[class_name][psize]["results"]
            ax.set_title(f"{class_name} ({psize}px)\nTop-1: {acc:.1f}%, Top-5: {top5:.1f}%")
            ax.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
show_patches(patch_dict)


We can see a clear difference between patches of different classes and sizes. In the smallest size, $32\times 32$ pixels, some of the patches clearly resemble their class. For instance, the goldfish patch clearly shows a goldfish. The eye and the color are very characteristic of the class. Overall, the patches with $32$ pixels have very strong colors that are typical for their class (yellow school bus, pink lipstick, greenish pineapple). The larger the patch becomes, the more stretched the pattern becomes. For the goldfish, we can still spot regions that might represent eyes and the characteristic orange color, but it is not clearly a single fish anymore. For the pineapple, we might interpret the top part of the image as the leaves of pineapple fruit, but it is more abstract than our small patches. Nevertheless, we can easily spot the alignment of the patch to class, even on the largest scale.

Let's now look at the quantitative results.

In [ ]:
%%html
<!-- Some HTML code to increase font size in the following table -->
<style>
th {font-size: 120%;}
td {font-size: 120%;}
</style>

In [ ]:
import tabulate
from IPython.display import display, HTML

def show_table(top_1=True):
    i = 0 if top_1 else 1
    table = [[name] + [f"{patch_dict[name][psize]['results'][i]:4.2f}%" for psize in patch_sizes]
             for name in class_names]
    display(HTML(tabulate.tabulate(
        table,
        tablefmt='html',
        headers=["Class name"] + [f"Patch size {psize}x{psize}" for psize in patch_sizes]
    )))


First, we will create a table of top-1 accuracy, meaning that how many images have been classified with the target class as highest prediction?

In [ ]:
show_table(top_1=True)

The clear trend, that we would also have expected, is that the larger the patch, the easier it is to fool the model. For the largest patch size of $64\times 64$, we are able to fool the model on almost all images, despite the patch covering only 8% of the image. The smallest patch actually covers 2% of the image, which is almost neglectable. Still, the fooling accuracy is quite remarkable. A large variation can be however seen across classes. While *school bus* and *pineapple* seem to be classes that were easily predicted, *toaster* and *lipstick* seem to be much harder for creating a patch. It is hard to intuitively explain why our patches underperform on those classes. Nonetheless, a fooling accuracy of >40% is still very good for such a tiny patch.

Let's also take a look at the top-5 accuracy:

In [ ]:
show_table(top_1=False)

We see a very similar pattern across classes and patch sizes. The patch size $64$ obtains >99.7% top-5 accuracy for any class, showing that we can almost fool the network on any image. A top-5 accuracy of >70% for the hard classes and small patches is still impressive and shows how vulnerable deep CNNs are to such attacks.

Finally, let's create some example visualizations of the patch attack in action.

# 10 Points

In [ ]:
def perform_patch_attack(patch, model=resnet34, dataset=full_dataset, class_names=None, location='bottom_right', alpha=0.9, device='cuda'):
    """
    Selects a random image from the dataset, applies a given patch, and visualizes the model's prediction.

    Args:
        patch (Tensor): Trained adversarial patch (normalized)
        model (nn.Module): Classification model
        dataset (Dataset): Dataset to sample from
        class_names (list): List of class labels
        location (str): Location to place the patch ("center" or "bottom_right")
        alpha (float): Transparency factor for blending
        device (str): 'cuda' or 'cpu'
    """
    import matplotlib.pyplot as plt
    from torchvision.transforms.functional import normalize

    model.eval()
    model.to(device)

    # Select a random sample
    img, label = random.choice(dataset)
    img = img.to(device)

    # Apply patch
    patched_img = img.clone()
    _, H, W = patched_img.shape
    _, h, w = patch.shape

    top = (H - h) // 2
    left = (W - w) // 2

    patched_img[:, top:top + h, left:left + w] = (
        (1 - alpha) * patched_img[:, top:top + h, left:left + w] + alpha * patch
    )

    # Run through model
    with torch.no_grad():
        output = model(patched_img.unsqueeze(0))
        probs = F.softmax(output, dim=1)
        top5_probs, top5_indices = torch.topk(probs, 5, dim=1)

    top5_probs = top5_probs[0].cpu().numpy()
    top5_indices = top5_indices[0].cpu().numpy()
    top5_labels = [class_names[i] if class_names else str(i) for i in top5_indices]

    # Inverse normalization
    inv_norm = transforms.Normalize(
        mean=[-m / s for m, s in zip(NORM_MEAN, NORM_STD)],
        std=[1 / s for s in NORM_STD]
    )
    img_vis = inv_norm(patched_img.cpu()).clamp(0, 1).permute(1, 2, 0).numpy()

    # Plot
    plt.figure(figsize=(6, 6))
    plt.imshow(img_vis)
    plt.axis('off')
    plt.title("Patched Image")
    plt.show()

    print(f"🎯 True Label: {class_names[label] if class_names else label}")
    print("🔍 Top-5 predictions:")
    for lbl, prob in zip(top5_labels, top5_probs):
        print(f"  → {lbl}: {prob * 100:.2f}%")


In [ ]:
perform_patch_attack(patch_dict['goldfish'][128]['patch'])

The tiny goldfish patch can change all of the predictions to "goldfish" as top class. Note that the patch attacks work especially well if the input image is semantically similar to the target class (e.g. a fish and the target class "goldfish" works better than an airplane image with that patch). Nevertheless, we can also let the network predict semantically dis-similar classes by using a larger patch:

In [ ]:
perform_patch_attack(patch_dict['school bus'][64]['patch'])

Although none of the images have anything to do with an American school bus, the high confidence of often 100% shows how powerful such attacks can be. With a few lines of code and access to the model, we were able to generate patches that we add to any image to make the model predict any class we want.

### Transferability of white-box attacks

FGSM and the adversarial patch attack were both focused on one specific image. However, can we transfer those attacks to other models? The adversarial patch attack as proposed in [Brown et al.](https://arxiv.org/pdf/1712.09665.pdf), was originally trained on multiple models, and hence was also able to work on many different network architecture. But how different are the patches for different models anyway? For instance, let's evaluate some of our patches trained above on a different network, e.g. DenseNet121.

# 15 Points

In [ ]:
from torchvision.models import densenet121

# Load DenseNet121 pretrained and freeze parameters
densenet = densenet121(weights='IMAGENET1K_V1')
for param in densenet.parameters():
    param.requires_grad = False

# Replace final classifier
num_classes = len(full_dataset.classes)
densenet.classifier = torch.nn.Linear(densenet.classifier.in_features, num_classes)
densenet = densenet.to(device)


Feel free to change the class name and/or patch size below to test out different patches.

In [ ]:
class_name = 'pineapple'
patch_size = 64
print(f"🔁 Transferring patch \"{class_name}\" of size {patch_size}x{patch_size}")

# Load patch from your dictionary
patch_tensor = patch_dict[class_name][patch_size]["patch"]

# Get class index using synset
target_syn = name_to_synset[class_name]
target_idx = full_dataset.class_to_idx[target_syn]

# Evaluate patch on the new transfer model (e.g. DenseNet)
top1, top5 = eval_patch(
    model=densenet,
    patch=patch_tensor,
    data_loader=test_loader,
    target_class_name=class_name,
    name_to_synset=name_to_synset,
    full_dataset=full_dataset,
    device=device
)

# Print results
print(f"🎯 Transfer Top-1 Fooling Accuracy: { top1:.2f}%")
print(f"🎯 Transfer Top-5 Fooling Accuracy: {top5:.2f}%")


Although the fool accuracy is significantly lower than on the original ResNet34, it still has a considerable impact on DenseNet although the networks have completely different architectures and weights. If you would compare more patches and models, some would work better than others. However, one aspect which allows patch attacks to generalize well is if all the networks have been trained on the same data. In this case, all networks have been trained on ImageNet. Dataset biases make the networks recognize specific patterns in the underlying image data that humans would not have seen, and/or only work for the given dataset. This is why the knowledge of what data has been used to train a specific model is already worth a lot in the context of adversarial attacks.

## Protecting against adversarial attacks

There are many more attack strategies than just FGSM and adversarial patches that we haven't discussed and implemented ourselves here. However, what about the other perspective? What can we do to *protect* a network against adversarial attacks? The sad truth to this is: not much.

White-box attacks require access to the model and its gradient calculation. The easiest way of preventing this is by ensuring safe, private storage of the model and its weights. However, some attacks, called black-box attacks, also work without access to the model's parameters, or white-box attacks can also generalize as we have seen above on our short test on transferability.

So, how could we eventually protect a model? An intuitive approach would to train/finetune a model on such adversarial images, leading to an adversarial training similar to a GAN. During training, we would pretend to be the attacker, and use for example FGSM as an augmentation strategy. However, this usually just ends up in an oscillation of the defending network between weak spots. Another common trick to increase robustness against adversarial attacks is defensive distillation ([Papernot et al.](https://arxiv.org/pdf/1511.04508.pdf)). Instead of training the model on the dataset labels, we train a secondary model on the softmax predictions of the first one. This way, the loss surface is "smoothed" in the directions an attacker might try to exploit, and it becomes more difficult for the attacker to find adversarial examples. Nevertheless, there hasn't been found the one, true strategy that works against all possible adversarial attacks.

Why are CNNs, or neural networks in general, so vulnerable to adversarial attacks? While there are many possible explanations, the most intuitive is that neural networks don't know what they don't know. Even a large dataset represents just a few sparse points in the extremely large space of possible images. A lot of the input space has not been seen by the network during training, and hence, we cannot guarantee that the prediction for those images is any useful. The network instead learns a very good classification on a smaller region, often referred to as manifold, while ignoring the points outside of it. NNs with uncertainty prediction could potentially help to discover what the network does not know.
Another possible explanation lies in the activation function. As we know, most CNNs use ReLU-based activation functions. While those have enabled great success in training deep neural networks due to their stable gradient for positive values, they also constitute a possible flaw. The output range of a ReLU neuron can be arbitrarily high. Thus, if we design a patch or the noise in the image to cause a very high value for a single neuron, it can overpower many other features in the network. Thus, although ReLU stabilizes training, it also offers a potential point of attack for adversaries.

# 2.black box attack

A black-box attack in the context of attacking Automatic Speech Recognition (ASR) models refers to an adversarial attack where the attacker has no knowledge of the model's internal architecture, parameters, or training data. Instead, the attacker can only interact with the model by sending inputs (speech/audio) and observing outputs (transcriptions).

Key Characteristics of Black-Box Attacks on ASR Models:
No Access to Model Internals

The attacker cannot see the model's weights, gradients, or architecture.

The attack relies solely on input-output queries (e.g., sending audio and checking transcriptions).

Adversarial Examples Are Crafted Based on Output Feedback

The attacker tries small perturbations (e.g., noise, distortions, or subtle signal modifications) and observes how the transcription changes.

Techniques like evolutionary optimization or gradient estimation (if limited queries are allowed) may be used.

# Common Attack Goals

Targeted Attack: Force the ASR to transcribe a specific malicious phrase (e.g., "Open the door" → "Transfer $1000").

Untargeted Attack: Cause mis-transcription to degrade performance (e.g., turning speech into gibberish).

Denial-of-Service (DoS): Overload the system with adversarial noise to make it unusable.

# Real-World Feasibility

Some attacks work even over-the-air (e.g., playing modified audio through a speaker).

Example: "Hidden Voice Commands" (Carlini et al.) showed that small perturbations could trick ASR systems into hearing unintended commands.

Common Black-Box Attack Techniques on ASR Models:
Genetic Algorithms / Evolutionary Strategies (Optimize perturbations without gradients).

Query-Based Optimization (Estimate gradients by repeatedly querying the model).

Transfer Attacks (Use a surrogate model to craft adversarial samples that transfer to the target model).

Universal Perturbations (Find a noise pattern that fools the model on multiple inputs).

Defenses Against Black-Box ASR Attacks:
Input Filtering (Detect adversarial perturbations).

Randomized Smoothing (Add noise to inputs to make attacks harder).

Adversarial Training (Train the model on perturbed samples to improve robustness).

# Install necessary library

In [ ]:
!pip install torch torchaudio librosa matplotlib numpy omegaconf pandas SoundFile speechbrain hydra-core jiwer tqdm

in this question we will design query-efficient black-box attack method  to fool Automatic Speech Recognition (ASR) systems by leveraging a neural predictor to estimate adversarial perturbations without requiring access to the target model's internals. Unlike traditional black-box attacks that rely on brute-force optimization or gradient estimation, NP-Attack trains a surrogate model (the "neural predictor") to predict adversarial perturbations, significantly reducing the number of queries needed to generate successful attacks.

# 5 Points

# Set the config

TODO: Experiments  to Modify & Observe Changes
Students should tweak these parameters and observe how they affect attack success rate, WER, and audio quality.

1. Change Attack Strength (eps_perb)
Default: eps_perb: 0.00 (no attack)

Try: 0.01, 0.05, 0.1

Expected Change:

Higher eps_perb → More distortion in audio → Higher WER (attack succeeds).

Too high eps_perb → Audio becomes unintelligible.

2. Modify Computational Budget (budget)
Default: budget: 5000

Try: 1000, 5000, 8000

Expected Change:

Higher budget → More iterations → Better attack success (but slower).

Lower budget → Faster but may not reach min_wer.

3. Change Norm Constraint (norm)
Default: norm: inf (L∞ norm, max perturbation per sample)

Try: norm: 2 (L2 norm, Euclidean distance)

Expected Change:

inf → Focuses on worst-case perturbations.

2 → Smoother, less noticeable distortions.

4. Adjust Target WER (min_wer)
Default: min_wer: 1e-9 (almost 0% WER)

Try: 0.1, 0.5 (10%, 50% WER)

Expected Change:

Higher min_wer → Attack stops earlier (faster but less effective).

5. Use a Different Audio File (wave_file)
Default: 237-134500-0001.flac

Try: Replace with another .flac file (e.g., 8455-210777-0068.flac from LibriSpeech).

Expected Change:

Some files may be easier/harder to attack due to speaker variability.

6. Enable Logging (out: True, hydra.verbose: True)
Default: out: False, verbose: false

Try: Enable them to see debug info.

Expected Change:

More detailed logs help understand attack progress.

# loading model

In [ ]:
!pip install speechbrain torch torchaudio librosa soundfile


In [ ]:
!pip install torch torchaudio librosa matplotlib numpy omegaconf pandas SoundFile speechbrain hydra-core jiwer tqdm


In [ ]:
import torch
from speechbrain.pretrained import EncoderDecoderASR

class ASR:
    def __init__(self, hp, device) -> None:
        self.hp = hp
        self.model = EncoderDecoderASR.from_hparams(
            source=hp.source,
            savedir=hp.savedir,
            run_opts={"device": str(device)},
            freeze_params=True
        )
        self.b_len = torch.tensor([1.]).to(device)

    def transcribe(self, wave):
        wave = torch.tensor(wave).to(self.model.device).unsqueeze(0)
        pred = self.model.transcribe_batch(wave, self.b_len)[0][0]
        return pred

    def transcribe_file(self, file_path):  # 👈 این تابع رو اضافه کن
        return self.model.transcribe_file(file_path)



# predictor

# 35 Points

Implement and experiment with different neural network architectures (MLP, CNN, Predictor) to understand their behavior in adversarial settings. we will modify hyperparameters, train models, and analyze results.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

def wn_conv1d(*args, **kwargs):
    return weight_norm(nn.Conv1d(*args, **kwargs))

def weight_reset(m):
    if isinstance(m, nn.Linear):
        m.reset_parameters()

class Audio2Spec(nn.Module):
    """Waveform to spectrogram."""
    def __init__(self, n_fft=512, hop_length=256, win_length=512):
        super().__init__()
        window = torch.hann_window(win_length).float()
        self.register_buffer("window", window)
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length

    def forward(self, audio):
        spec = torch.stft(audio, n_fft=self.n_fft, hop_length=self.hop_length,
                          win_length=self.win_length, window=self.window,
                          return_complex=True)
        spec_mag = torch.abs(spec)
        return spec_mag

class MLP(nn.Module):
    def __init__(self, hp, device):
        super().__init__()
        input_dim = hp.input_dim
        hidden_dim = hp.hidden_dim
        output_dim = hp.output_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        self.to(device)

    def forward(self, x):
        return self.net(x)

class CNN(nn.Module):
    def __init__(self, hp, device):
        super().__init__()
        self.net = nn.Sequential(
            wn_conv1d(hp.input_channels, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            wn_conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear(128 * (hp.input_length // 4), hp.output_dim)
        )
        self.to(device)

    def forward(self, x):
        return self.net(x)

class Predictor(nn.Module):
    def __init__(self, hp, device):
        super().__init__()
        self.hp = hp
        self.device = device
        self.net = MLP(hp, device) if hp.model_type == 'MLP' else CNN(hp, device)

    def forward(self, x):
        return self.net(x)

    def fit(self, x_train, y_train, epochs=10, batch_size=64):
        dataset = TensorDataset(torch.tensor(x_train, dtype=torch.float32),
                                torch.tensor(y_train, dtype=torch.float32))
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = torch.optim.Adam(self.parameters(), lr=self.hp.lr)
        loss_fn = nn.MSELoss()

        self.train()
        for epoch in range(epochs):
            total_loss = 0
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                pred = self(xb)
                loss = loss_fn(pred, yb)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(loader):.4f}")

    def optim_inputs(self, x_init, steps=100, lr=1e-2):
        self.eval()
        x_adv = torch.tensor(x_init, dtype=torch.float32, requires_grad=True, device=self.device)
        optimizer = torch.optim.Adam([x_adv], lr=lr)

        for step in range(steps):
            pred = self(x_adv)
            loss = -torch.norm(pred, p=2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if step % 10 == 0:
                print(f"Step {step}/{steps} | Loss: {loss.item():.4f}")

        return x_adv.detach().cpu().numpy()


# attacker

# Attack Overview
#Goal:

Generate adversarial audio samples that force an ASR system to produce incorrect transcriptions (either targeted or untargeted).

#Constraints:

No access to the ASR model’s architecture, weights, or gradients (strict black-box setting).

Limited queries to avoid detection (unlike genetic algorithms or gradient estimation, which require many queries).

# 25 Points

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import librosa as lr
from jiwer import wer
from pathlib import Path

In [ ]:
class NPAttacker:
    def __init__(self, hp):
        self.hp = hp
        self.asr = ASR(hp.asr, torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        self.rng = np.random.RandomState(hp.seed)
        self.norm = np.inf if hp.norm == 'inf' else 2
        self.sample_id = Path(hp.wave_file).stem
        self.pre = Predictor(hp.strategy.predictor, torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

    def attack(self, wave_file):
        x, sr = lr.load(wave_file, sr=self.hp.sr)
        y_true = self.asr.transcribe(x)
        print(f"Original Transcription: {y_true}")

        for _ in range(self.hp.budget):
            theta = self.get_theta(self.rng.randn(*x.shape))
            dist = self.b_dist(theta)
            if dist is not None:
                adv_wave = self.get_wave(theta, dist)
                y_pred = self.asr.transcribe(adv_wave)
                print(f"Attempted Attack. Prediction: {y_pred}")
                if wer(y_true, y_pred) >= self.hp.min_wer:
                    print("✅ Attack succeeded!")
                    return adv_wave
        print("❌ Attack failed.")
        return None

    def get_theta(self, x):
        if self.norm == np.inf:
            return np.sign(x)
        else:
            return x / (np.linalg.norm(x) + 1e-10)

    def get_wave(self, theta, mag):
        x, _ = lr.load(self.hp.wave_file, sr=self.hp.sr)
        pert = theta * mag
        adv = np.clip(x + pert, -1.0, 1.0)
        return adv

    def query(self, theta, mag):
        adv = self.get_wave(theta, mag)
        y_true = self.asr.transcribe_file(self.hp.wave_file)
        y_pred = self.asr.transcribe(adv)
        return wer(y_true, y_pred) >= self.hp.min_wer

    def b_dist(self, x, tol=1e-4, incr=0.01):
        mag = 0.0
        while not self.query(x, mag):
            mag += incr
            if mag > self.hp.eps_perb:
                return None

        low, high = 0, mag
        while (high - low) > tol:
            mid = (low + high) / 2
            if self.query(x, mid):
                high = mid
            else:
                low = mid
        return high

    def eval_attack(self, ae):
        import soundfile as sf
        filename = f"adv_{self.sample_id}.wav"
        sf.write(filename, ae, self.hp.sr)
        print(f"Adversarial example saved as {filename}")

In [ ]:
# نصب پکیج‌ها
!pip install torch torchaudio librosa matplotlib numpy omegaconf pandas SoundFile speechbrain hydra-core jiwer tqdm

# دانلود دیتاست LibriSpeech
!wget -nc http://www.openslr.org/resources/12/test-clean.tar.gz
!tar -xvzf test-clean.tar.gz

# انتخاب یک فایل صوتی
import os

sample_audio = None
for root, dirs, files in os.walk("LibriSpeech/test-clean/"):
    for file in files:
        if file.endswith(".flac"):
            sample_audio = os.path.join(root, file)
            break
    if sample_audio:
        break
import soundfile as sf
y, sr = lr.load(sample_audio, sr=16000)
output_path = "./data/original.wav"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
sf.write(output_path, y, sr)
print(f"✅ فایل ذخیره شد در: {output_path}")



In [ ]:
import torch
import librosa as lr
import numpy as np
import soundfile as sf
from types import SimpleNamespace
from tqdm import tqdm
from IPython.display import Audio

# 🎛 Spectrogram extractor
class Audio2Spec(torch.nn.Module):
    def __init__(self, n_fft=512, hop_length=256, win_length=512):
        super().__init__()
        window = torch.hann_window(win_length)
        self.register_buffer("window", window)
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length

    def forward(self, x):
        spec = torch.stft(x, n_fft=self.n_fft, hop_length=self.hop_length,
                          win_length=self.win_length, window=self.window,
                          return_complex=True)
        return torch.abs(spec)

# 🎵 Load audio
wave_file = "/content/data/original.wav"
wave, sr = lr.load(wave_file, sr=16000)
wave_tensor = torch.tensor(wave)

# 🎛 Extract spectrogram and flatten for MLP
spec_layer = Audio2Spec()
spec = spec_layer(wave_tensor)  # [257, T]
spec = spec.T  # [T, 257]
flat_input = spec.flatten().unsqueeze(0)  # [1, T*257]

X = flat_input.clone()
Y = flat_input.clone()

# ✅ Define hyperparameters for MLP
hp = SimpleNamespace(
    input_dim=X.shape[1],
    hidden_dim=1024,
    output_dim=X.shape[1],
    lr=0.001,
    model_type='MLP'
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✅ Define MLP model
class MLP(torch.nn.Module):
    def __init__(self, hp, device):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(hp.input_dim, hp.hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hp.hidden_dim, hp.hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hp.hidden_dim, hp.output_dim)
        )
        self.to(device)

    def forward(self, x):
        return self.net(x)

class Predictor(torch.nn.Module):
    def __init__(self, hp, device):
        super().__init__()
        self.hp = hp
        self.device = device
        self.net = MLP(hp, device)

    def forward(self, x):
        return self.net(x)

    def fit(self, x_train, y_train, epochs=5, batch_size=1):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hp.lr)
        loss_fn = torch.nn.MSELoss()
        self.train()
        for epoch in range(epochs):
            pred = self(x_train.to(self.device))
            loss = loss_fn(pred, y_train.to(self.device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

# ✅ Train predictor
model = Predictor(hp, device)
model.fit(X, Y, epochs=5)

# ✅ Save original audio
sf.write("original.wav", wave, sr)
print("✅ فایل صوتی اصلی ذخیره شد.")


In [ ]:
from pathlib import Path
from jiwer import wer
from speechbrain.pretrained import EncoderDecoderASR

# ✅ ASR wrapper
class ASR:
    def __init__(self, hp, device) -> None:
        self.hp = hp
        self.model = EncoderDecoderASR.from_hparams(
            source=hp.source,
            savedir=hp.savedir,
            run_opts={"device": str(device)},
            freeze_params=True
        )
        self.b_len = torch.tensor([1.]).to(device)

    def transcribe(self, wave):
        wave = torch.tensor(wave).to(self.model.device).unsqueeze(0)
        pred = self.model.transcribe_batch(wave, self.b_len)[0][0]
        return pred

    def transcribe_file(self, file_path):
        return self.model.transcribe_file(file_path)

# ✅ NP-Attack class
class NPAttacker:
    def __init__(self, hp, predictor_model):
        self.hp = hp
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.asr = ASR(hp.asr, self.device)
        self.predictor = predictor_model.to(self.device)
        self.predictor.eval()
        self.rng = np.random.RandomState(hp.seed)
        self.norm = np.inf if hp.norm == 'inf' else 2
        self.sample_id = Path(hp.wave_file).stem

    def attack(self, wave_file):
        x, sr = lr.load(wave_file, sr=self.hp.sr)
        y_true = self.asr.transcribe(x)
        print(f"🎧 Original Transcription: {y_true}")

        for i in tqdm(range(self.hp.budget), desc="🔁 Attacking", ncols=80):
            theta = self.get_theta(self.rng.randn(*x.shape))
            dist = self.b_dist(theta)
            if dist is not None:
                adv_wave = self.get_wave(theta, dist)
                y_pred = self.asr.transcribe(adv_wave)
                wer_score = wer(y_true, y_pred)
                print(f"[{i}] WER: {wer_score:.2f} | Pred: {y_pred}")
                if wer_score >= self.hp.min_wer:
                    print("✅ Attack succeeded!")
                    return adv_wave
        print("❌ Attack failed.")
        return None

    def get_theta(self, x):
        return np.sign(x) if self.norm == np.inf else x / (np.linalg.norm(x) + 1e-10)

    def get_wave(self, theta, mag):
        x, _ = lr.load(self.hp.wave_file, sr=self.hp.sr)
        pert = theta * mag
        adv = np.clip(x + pert, -1.0, 1.0)
        return adv

    def query(self, theta, mag):
        adv = self.get_wave(theta, mag)
        y_true = self.asr.transcribe_file(self.hp.wave_file)
        y_pred = self.asr.transcribe(adv)
        return wer(y_true, y_pred) >= self.hp.min_wer

    def b_dist(self, x, tol=1e-4, incr=0.01):
        mag = 0.0
        while not self.query(x, mag):
            mag += incr
            if mag > self.hp.eps_perb:
                return None
        low, high = 0, mag
        while (high - low) > tol:
            mid = (low + high) / 2
            if self.query(x, mid):
                high = mid
            else:
                low = mid
        return high

# ✅ Config setup
hp_attack = SimpleNamespace(
    sr=16000,
    seed=42,
    budget=50,
    eps_perb=0.03,
    min_wer=0.3,
    norm="inf",
    wave_file="original.wav",
    asr=SimpleNamespace(
        source="speechbrain/asr-crdnn-rnnlm-librispeech",
        savedir="pretrained_models/asr-crdnn-rnnlm-librispeech"
    )
)

# ✅ اجرای حمله با مدل آموزش‌دیده MLP
attacker = NPAttacker(hp_attack, model)
adv_wave = attacker.attack(hp_attack.wave_file)

# ✅ ذخیره خروجی حمله
if adv_wave is not None:
    out_path = "adv.wav"
    sf.write(out_path, adv_wave, hp_attack.sr)
    print("✅ فایل adversarial ذخیره شد:", out_path)
    print("📝 ترنسکریپشن نهایی:", attacker.asr.transcribe(adv_wave))
    display(Audio("original.wav"))
    display(Audio(out_path))
else:
    print("❌ حمله موفق نبود.")


In [ ]:
# نصب
!pip install librosa soundfile tqdm

# ایمپورت
import librosa as lr
import librosa.display
import soundfile as sf
import numpy as np
import torch
from types import SimpleNamespace
from matplotlib import pyplot as plt

# فرض: کلاس‌های MLP، CNN، Predictor، Audio2Spec از قبل اجرا شده‌اند

# ----------------------------------------------------
# 1. بارگذاری فایل صوتی و ساخت spectrogram
# ----------------------------------------------------
wave_file = "/content/Recording (9).wav"
x, sr = lr.load(wave_file, sr=16000)
print(f"✅ Wave loaded: {x.shape}, sample rate: {sr}")

# تبدیل به spectrogram
spec_extractor = Audio2Spec(n_fft=512, hop_length=256, win_length=512)
with torch.no_grad():
    spec = spec_extractor(torch.tensor(x))
    spec_np = spec.numpy()  # (freq, time)

# reshape برای مدل MLP
X = spec_np.T  # (time, freq)
Y = X.copy()

print(f"✅ Spectrogram shape: {spec_np.shape}")
print(f"✅ X shape (time, freq): {X.shape}")

# ----------------------------------------------------
# 2. ساخت مدل و آموزش
# ----------------------------------------------------
hp = SimpleNamespace(
    input_dim=X.shape[1],
    hidden_dim=512,
    output_dim=X.shape[1],
    input_channels=1,
    input_length=1000,
    lr=0.0001,
    model_type='MLP'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Predictor(hp, device)
model.fit(X, Y, epochs=300, batch_size=32)

# ----------------------------------------------------
# 3. پیش‌بینی روی نمونه کامل
# ----------------------------------------------------
with torch.no_grad():
    input_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    output_tensor = model(input_tensor).cpu().numpy()  # (time, freq)

# ----------------------------------------------------
# 4. بازسازی صوت از output با Griffin-Lim
# ----------------------------------------------------
reconstructed_audio = lr.feature.inverse.griffinlim(output_tensor.T,
                                                    n_iter=32,
                                                    hop_length=256,
                                                    win_length=512)

sf.write("model_output.wav", reconstructed_audio, sr)
print("✅ خروجی مدل ذخیره شد: model_output.wav")

# ----------------------------------------------------
# 5. مقایسه تصویری spectrogram‌ها
# ----------------------------------------------------
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
librosa.display.specshow(X.T, sr=sr, hop_length=256, x_axis='time', y_axis='hz')
plt.colorbar(format="%+2.f dB")
plt.title("🎧 Original Spectrogram")

plt.subplot(1, 2, 2)
librosa.display.specshow(output_tensor.T, sr=sr, hop_length=256, x_axis='time', y_axis='hz')
plt.colorbar(format="%+2.f dB")
plt.title("🧠 Reconstructed Spectrogram")

plt.tight_layout()
plt.show()

# ----------------------------------------------------
# 6. پخش فایل صوتی اصلی و بازسازی‌شده
# ----------------------------------------------------
from IPython.display import Audio

print("🎧 صدای اصلی:")
display(Audio(wave_file))

print("🧠 صدای بازسازی‌شده:")
display(Audio("model_output.wav"))


## References

[1] Goodfellow, Ian J., Jonathon Shlens, and Christian Szegedy. "Explaining and harnessing adversarial examples." ICLR 2015.

[2] Hendrik Metzen, Jan, et al. "Universal adversarial perturbations against semantic image segmentation." Proceedings of the IEEE International Conference on Computer Vision. 2017.

[3] Anant Jain. "Breaking neural networks with adversarial attacks." [Blog post](https://towardsdatascience.com/breaking-neural-networks-with-adversarial-attacks-f4290a9a45aa) 2019.

[4] Mirco Ravanelli and Titouan Parcollet "SpeechBrain: A General-Purpose Speech Toolkit"